In [1]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("lovishbansal123/netflix-dataset")

# print("Path to dataset files:", path)

In [ ]:
import pandas as pd

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv('../dataset/netflix_titles.csv', delimiter=',', encoding='utf-8')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [4]:
df.fillna({'date_added': f'January 1, {df["release_year"]+1}'}, inplace=True) # дополняем недостающие значения
df['date_added'].isna().value_counts()

date_added
False    8807
Name: count, dtype: int64

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8807 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [6]:
df[['director', 'cast', 'country', 'rating', 'duration']] = df[['director', 'cast', 'country', 'rating', 'duration']].fillna('Unknown') # дополняем недостающие значения

In [7]:
# инициализируем векторизатор
tfidf = TfidfVectorizer(stop_words='english')
# преобразовываем описание фильма в векторную матрицу, которую далее будем использовать
tfidf_matrix = tfidf.fit_transform(df['description'])
tfidf_matrix.shape

(8807, 18895)

In [8]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix) # используем косинусное расстояние для присовения "индекса похожести"
cosine_sim.shape

(8807, 8807)

In [9]:
# делаем из нашей матрицы датафрейм
cosine_sim_df = pd.DataFrame(data=cosine_sim, index=df['title'], columns=df['title'])
cosine_sim_df.head()

title,Dick Johnson Is Dead,Blood & Water,Ganglands,Jailbirds New Orleans,Kota Factory,Midnight Mass,My Little Pony: A New Generation,Sankofa,The Great British Baking Show,The Starling,...,Zak Storm,Zed Plus,Zenda,Zindagi Gulzar Hai,Zinzana,Zodiac,Zombie Dumb,Zombieland,Zoom,Zubaan
title,,,,,,,,,,,,,,,,,,,,,
Dick Johnson Is Dead,1.000000,0.0,0.0,0.0,0.015222,0.000000,0.00000,0.0,0.040165,0.017411,...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.000000,0.015383,0.000000
Blood & Water,0.000000,1.0,0.0,0.0,0.000000,0.031227,0.05166,0.0,0.000000,0.000000,...,0.035331,0.032784,0.116936,0.0,0.032609,0.0,0.042,0.000000,0.000000,0.000000
Ganglands,0.000000,0.0,1.0,0.0,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.000000,0.000000,0.022301
Jailbirds New Orleans,0.000000,0.0,0.0,1.0,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.000000,0.016386,0.000000
Kota Factory,0.015222,0.0,0.0,0.0,1.000000,0.000000,0.00000,0.0,0.000000,0.016718,...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.035196,0.068784,0.000000


In [ ]:
# # перезапись изначального файла и запись матрицы в файл для последующего использования
# df.to_csv('../dataset/netflix_titles.csv', index=False)
cosine_sim_df.to_csv('../dataset/cosine_sim.csv')